In [91]:
import os
os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['PATH'] = os.environ['PATH'] + r';C:\hadoop\bin'

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, count, avg, max, min, sum,
    upper, lower, length, concat_ws, trim,
    rank, dense_rank, row_number, lead, lag,
    round as spark_round
)
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("TelecomAnalytics") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

BRONZE = r"C:\new_env\hexaware-sql\june_13\capstone_1\bronze"
SILVER = r"C:\new_env\hexaware-sql\june_13\capstone_1\silver"
GOLD   = r"C:\new_env\hexaware-sql\june_13\capstone_1\gold"

In [92]:
# 1
customers_df = spark.read.csv(
    r"C:\new_env\hexaware-sql\june_13\capstone_1\customers.csv",
    header=True, inferSchema=True
)
customers_df.show(truncate=False)

+-----------+-------------+---------+-----------+---+------+-------+--------+
|customer_id|customer_name|city     |state      |age|gender|plan_id|status  |
+-----------+-------------+---------+-----------+---+------+-------+--------+
|101        |Rahul Sharma |Hyderabad|Telangana  |35 |Male  |P101   |Active  |
|102        |Priya Reddy  |Bangalore|Karnataka  |29 |Female|P102   |Active  |
|103        |Amit Kumar   |Mumbai   |Maharashtra|42 |Male  |P103   |Inactive|
|104        |Sneha Patel  |Chennai  |Tamil Nadu |31 |Female|P101   |Active  |
|105        |Farhan Ali   |Delhi    |Delhi      |55 |Male  |P104   |Active  |
|106        |Neha Singh   |Pune     |Maharashtra|38 |Female|P102   |Active  |
|107        |Arjun Verma  |Hyderabad|Telangana  |26 |Male  |P103   |Inactive|
|108        |Meera Nair   |Kochi    |Kerala     |48 |Female|P104   |Active  |
|109        |Kiran Rao    |Bangalore|Karnataka  |33 |Male  |P101   |Active  |
|110        |Nisha Reddy  |Delhi    |Delhi      |41 |Female|P102

In [93]:
# 2
usage_df = spark.read.csv(
    r"C:\new_env\hexaware-sql\june_13\capstone_1\usage.csv",
    header=True, inferSchema=True
)
usage_df.show(truncate=False)

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|usage_month        |data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|1001    |101        |2026-01-01 00:00:00|45          |900         |120      |
|1002    |102        |2026-01-01 00:00:00|30          |600         |80       |
|1003    |103        |2026-01-01 00:00:00|12          |250         |40       |
|1004    |104        |2026-01-01 00:00:00|55          |1100        |150      |
|1005    |105        |2026-01-01 00:00:00|75          |1500        |200      |
|1006    |106        |2026-01-01 00:00:00|28          |500         |60       |
|1007    |107        |2026-01-01 00:00:00|10          |200         |20       |
|1008    |108        |2026-01-01 00:00:00|80          |1600        |250      |
|1009    |109        |2026-01-01 00:00:00|48          |950         |100      |
|1010    |110        |2026-01-01 00:00:00|32        

In [94]:
# 3
payments_df = spark.read.csv(
    r"C:\new_env\hexaware-sql\june_13\capstone_1\payments.csv",
    header=True, inferSchema=True
)
payments_df.show(truncate=False)

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|bill_month         |amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|5001      |101        |2026-01-01 00:00:00|499        |UPI         |Success       |
|5002      |102        |2026-01-01 00:00:00|799        |Card        |Success       |
|5003      |103        |2026-01-01 00:00:00|299        |Cash        |Failed        |
|5004      |104        |2026-01-01 00:00:00|499        |UPI         |Success       |
|5005      |105        |2026-01-01 00:00:00|1199       |Card        |Success       |
|5006      |106        |2026-01-01 00:00:00|799        |UPI         |Success       |
|5007      |107        |2026-01-01 00:00:00|299        |Cash        |Pending       |
|5008      |108        |2026-01-01 00:00:00|1199       |Card        |Success       |
|5009      |109        |2026-01-01 00:00:00|499        |UPI      

In [95]:
# 4
plans_df = spark.read.option("multiline", "true").json(
    r"C:\new_env\hexaware-sql\june_13\capstone_1\plans.json"
)
plans_df.show(truncate=False)

+-------------+---------------------------+-----------+-------+------------+
|data_limit_gb|features                   |monthly_fee|plan_id|plan_name   |
+-------------+---------------------------+-----------+-------+------------+
|50           |{false, National, true}    |499        |P101   |Smart Basic |
|75           |{true, National, true}     |799        |P102   |Smart Plus  |
|25           |{false, NULL, false}       |299        |P103   |Budget Saver|
|100          |{true, International, true}|1199       |P104   |Premium Max |
+-------------+---------------------------+-----------+-------+------------+



In [96]:
# 5
customers_df.printSchema()
usage_df.printSchema()
payments_df.printSchema()
plans_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- plan_id: string (nullable = true)
 |-- status: string (nullable = true)

root
 |-- usage_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- usage_month: timestamp (nullable = true)
 |-- data_used_gb: integer (nullable = true)
 |-- call_minutes: integer (nullable = true)
 |-- sms_count: integer (nullable = true)

root
 |-- payment_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- bill_month: timestamp (nullable = true)
 |-- amount_paid: integer (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- payment_status: string (nullable = true)

root
 |-- data_limit_gb: long (nullable = true)
 |-- features: struct (nullable = true)
 |    |-- ott_included: boolean (nullable = true)
 |

In [97]:
# 6
print("Customers :", customers_df.count())
print("Usage     :", usage_df.count())
print("Payments  :", payments_df.count())
print("Plans     :", plans_df.count())

Customers : 12
Usage     : 15
Payments  : 15
Plans     : 4


In [98]:
# 7
customers_df.write.mode("overwrite").parquet(BRONZE + r"\customers")
usage_df.write.mode("overwrite").parquet(BRONZE + r"\usage")
payments_df.write.mode("overwrite").parquet(BRONZE + r"\payments")
plans_df.write.mode("overwrite").parquet(BRONZE + r"\plans")
print("Bronze Parquet saved")

Bronze Parquet saved


In [99]:
# 8
customers_df.filter(col('plan_id').isNull()).show(truncate=False)

+-----------+-------------+---------+---------+---+------+-------+------+
|customer_id|customer_name|city     |state    |age|gender|plan_id|status|
+-----------+-------------+---------+---------+---+------+-------+------+
|112        |Ayesha Khan  |Hyderabad|Telangana|28 |Female|NULL   |Active|
+-----------+-------------+---------+---------+---+------+-------+------+



In [100]:
# 9
usage_df.filter(col('data_used_gb').isNull()).show(truncate=False)

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|usage_month        |data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|1015    |105        |2026-02-01 00:00:00|NULL        |1450        |210      |
+--------+-----------+-------------------+------------+------------+---------+



In [101]:
# 10
payments_df.filter(col('amount_paid').isNull()).show(truncate=False)

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|bill_month         |amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|5011      |112        |2026-01-01 00:00:00|NULL       |UPI         |Success       |
+----------+-----------+-------------------+-----------+------------+--------------+



In [102]:
# 11
payments_df.filter(col('payment_mode').isNull()).show(truncate=False)

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|bill_month         |amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|5015      |105        |2026-02-01 00:00:00|1199       |NULL        |Pending       |
+----------+-----------+-------------------+-----------+------------+--------------+



In [103]:
# 12
usage_clean = usage_df.fillna({'data_used_gb': 0})
usage_clean.show(truncate=False)

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|usage_month        |data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|1001    |101        |2026-01-01 00:00:00|45          |900         |120      |
|1002    |102        |2026-01-01 00:00:00|30          |600         |80       |
|1003    |103        |2026-01-01 00:00:00|12          |250         |40       |
|1004    |104        |2026-01-01 00:00:00|55          |1100        |150      |
|1005    |105        |2026-01-01 00:00:00|75          |1500        |200      |
|1006    |106        |2026-01-01 00:00:00|28          |500         |60       |
|1007    |107        |2026-01-01 00:00:00|10          |200         |20       |
|1008    |108        |2026-01-01 00:00:00|80          |1600        |250      |
|1009    |109        |2026-01-01 00:00:00|48          |950         |100      |
|1010    |110        |2026-01-01 00:00:00|32        

In [104]:
# 13
payments_clean = payments_df.fillna({'amount_paid': 0})
payments_clean.show(truncate=False)

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|bill_month         |amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|5001      |101        |2026-01-01 00:00:00|499        |UPI         |Success       |
|5002      |102        |2026-01-01 00:00:00|799        |Card        |Success       |
|5003      |103        |2026-01-01 00:00:00|299        |Cash        |Failed        |
|5004      |104        |2026-01-01 00:00:00|499        |UPI         |Success       |
|5005      |105        |2026-01-01 00:00:00|1199       |Card        |Success       |
|5006      |106        |2026-01-01 00:00:00|799        |UPI         |Success       |
|5007      |107        |2026-01-01 00:00:00|299        |Cash        |Pending       |
|5008      |108        |2026-01-01 00:00:00|1199       |Card        |Success       |
|5009      |109        |2026-01-01 00:00:00|499        |UPI      

In [105]:
# 14
payments_clean = payments_df.fillna({
    'amount_paid': 0,
    'payment_mode': 'Not Provided'
})
payments_clean.show(truncate=False)

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|bill_month         |amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|5001      |101        |2026-01-01 00:00:00|499        |UPI         |Success       |
|5002      |102        |2026-01-01 00:00:00|799        |Card        |Success       |
|5003      |103        |2026-01-01 00:00:00|299        |Cash        |Failed        |
|5004      |104        |2026-01-01 00:00:00|499        |UPI         |Success       |
|5005      |105        |2026-01-01 00:00:00|1199       |Card        |Success       |
|5006      |106        |2026-01-01 00:00:00|799        |UPI         |Success       |
|5007      |107        |2026-01-01 00:00:00|299        |Cash        |Pending       |
|5008      |108        |2026-01-01 00:00:00|1199       |Card        |Success       |
|5009      |109        |2026-01-01 00:00:00|499        |UPI      

In [106]:
# 15
customers_clean = customers_df.fillna({'plan_id': 'UNKNOWN'})
customers_clean.show(truncate=False)

+-----------+-------------+---------+-----------+---+------+-------+--------+
|customer_id|customer_name|city     |state      |age|gender|plan_id|status  |
+-----------+-------------+---------+-----------+---+------+-------+--------+
|101        |Rahul Sharma |Hyderabad|Telangana  |35 |Male  |P101   |Active  |
|102        |Priya Reddy  |Bangalore|Karnataka  |29 |Female|P102   |Active  |
|103        |Amit Kumar   |Mumbai   |Maharashtra|42 |Male  |P103   |Inactive|
|104        |Sneha Patel  |Chennai  |Tamil Nadu |31 |Female|P101   |Active  |
|105        |Farhan Ali   |Delhi    |Delhi      |55 |Male  |P104   |Active  |
|106        |Neha Singh   |Pune     |Maharashtra|38 |Female|P102   |Active  |
|107        |Arjun Verma  |Hyderabad|Telangana  |26 |Male  |P103   |Inactive|
|108        |Meera Nair   |Kochi    |Kerala     |48 |Female|P104   |Active  |
|109        |Kiran Rao    |Bangalore|Karnataka  |33 |Male  |P101   |Active  |
|110        |Nisha Reddy  |Delhi    |Delhi      |41 |Female|P102

In [107]:
# 16
customers_clean = customers_df.fillna({'plan_id': 'UNKNOWN'}).withColumn(
    'data_quality_status',
    when(col('plan_id').isNull(), 'Incomplete').otherwise('Complete')
)
usage_clean = usage_df.fillna({'data_used_gb': 0}).withColumn(
    'data_quality_status',
    when(col('data_used_gb').isNull(), 'Incomplete').otherwise('Complete')
)
payments_clean = payments_df.fillna({
    'amount_paid': 0,
    'payment_mode': 'Not Provided'
}).withColumn(
    'data_quality_status',
    when(
        col('amount_paid').isNull() | col('payment_mode').isNull(),
        'Incomplete'
    ).otherwise('Complete')
)
customers_clean.select('customer_id', 'data_quality_status').show()
usage_clean.select('usage_id', 'data_quality_status').show()
payments_clean.select('payment_id', 'data_quality_status').show()

+-----------+-------------------+
|customer_id|data_quality_status|
+-----------+-------------------+
|        101|           Complete|
|        102|           Complete|
|        103|           Complete|
|        104|           Complete|
|        105|           Complete|
|        106|           Complete|
|        107|           Complete|
|        108|           Complete|
|        109|           Complete|
|        110|           Complete|
|        111|           Complete|
|        112|           Complete|
+-----------+-------------------+

+--------+-------------------+
|usage_id|data_quality_status|
+--------+-------------------+
|    1001|           Complete|
|    1002|           Complete|
|    1003|           Complete|
|    1004|           Complete|
|    1005|           Complete|
|    1006|           Complete|
|    1007|           Complete|
|    1008|           Complete|
|    1009|           Complete|
|    1010|           Complete|
|    1011|           Complete|
|    1012|           

In [108]:
# 17
customers_clean.write.mode("overwrite").parquet(SILVER + r"\customers")
usage_clean.write.mode("overwrite").parquet(SILVER + r"\usage")
payments_clean.write.mode("overwrite").parquet(SILVER + r"\payments")
print("Silver Parquet saved")

Silver Parquet saved


In [109]:
# 18
plans_flat = plans_df.select(
    "plan_id",
    "plan_name",
    "monthly_fee",
    "data_limit_gb",
    col("features.unlimited_calls").alias("unlimited_calls"),
    col("features.ott_included").alias("ott_included"),
    col("features.roaming").alias("roaming")
)
plans_flat.show(truncate=False)

+-------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|plan_name   |monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming      |
+-------+------------+-----------+-------------+---------------+------------+-------------+
|P101   |Smart Basic |499        |50           |true           |false       |National     |
|P102   |Smart Plus  |799        |75           |true           |true        |National     |
|P103   |Budget Saver|299        |25           |false          |false       |NULL         |
|P104   |Premium Max |1199       |100          |true           |true        |International|
+-------+------------+-----------+-------------+---------------+------------+-------------+



In [110]:
# 19
plans_flat.select('plan_id', 'plan_name', 'unlimited_calls').show()

+-------+------------+---------------+
|plan_id|   plan_name|unlimited_calls|
+-------+------------+---------------+
|   P101| Smart Basic|           true|
|   P102|  Smart Plus|           true|
|   P103|Budget Saver|          false|
|   P104| Premium Max|           true|
+-------+------------+---------------+



In [111]:
# 20
plans_flat.select('plan_id', 'plan_name', 'ott_included').show()

+-------+------------+------------+
|plan_id|   plan_name|ott_included|
+-------+------------+------------+
|   P101| Smart Basic|       false|
|   P102|  Smart Plus|        true|
|   P103|Budget Saver|       false|
|   P104| Premium Max|        true|
+-------+------------+------------+



In [112]:
# 21
plans_flat.select('plan_id', 'plan_name', 'roaming').show()

+-------+------------+-------------+
|plan_id|   plan_name|      roaming|
+-------+------------+-------------+
|   P101| Smart Basic|     National|
|   P102|  Smart Plus|     National|
|   P103|Budget Saver|         NULL|
|   P104| Premium Max|International|
+-------+------------+-------------+



In [113]:
# 22
plans_flat = plans_flat.fillna({'roaming': 'Not Available'})
plans_flat.show(truncate=False)

+-------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|plan_name   |monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming      |
+-------+------------+-----------+-------------+---------------+------------+-------------+
|P101   |Smart Basic |499        |50           |true           |false       |National     |
|P102   |Smart Plus  |799        |75           |true           |true        |National     |
|P103   |Budget Saver|299        |25           |false          |false       |Not Available|
|P104   |Premium Max |1199       |100          |true           |true        |International|
+-------+------------+-----------+-------------+---------------+------------+-------------+



In [114]:
# 23
plans_flat.write.mode("overwrite").parquet(SILVER + r"\plans_flat")
print("Flattened plans saved")

Flattened plans saved


In [115]:
# 24
cust_plans = customers_clean.join(
    plans_flat, on='plan_id', how='left'
)
cust_plans.show(truncate=False)

+-------+-----------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|customer_id|customer_name|city     |state      |age|gender|status  |data_quality_status|plan_name   |monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming      |
+-------+-----------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+
|P101   |101        |Rahul Sharma |Hyderabad|Telangana  |35 |Male  |Active  |Complete           |Smart Basic |499        |50           |true           |false       |National     |
|P102   |102        |Priya Reddy  |Bangalore|Karnataka  |29 |Female|Active  |Complete           |Smart Plus  |799        |75           |true           |true        |National     |
|P103   |103        |Amit Kumar   |Mumbai   |Maharashtra|42 |Male  |Inactive|Complete           |Bud

In [116]:
# 25
cust_usage = customers_clean.join(
    usage_clean, on='customer_id', how='left'
)
cust_usage.show(truncate=False)

+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+--------+-------------------+------------+------------+---------+-------------------+
|customer_id|customer_name|city     |state      |age|gender|plan_id|status  |data_quality_status|usage_id|usage_month        |data_used_gb|call_minutes|sms_count|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+--------+-------------------+------------+------------+---------+-------------------+
|101        |Rahul Sharma |Hyderabad|Telangana  |35 |Male  |P101   |Active  |Complete           |1012    |2026-02-01 00:00:00|50          |1000        |130      |Complete           |
|101        |Rahul Sharma |Hyderabad|Telangana  |35 |Male  |P101   |Active  |Complete           |1001    |2026-01-01 00:00:00|45          |900         |120      |Complete           |
|102        |Priya Reddy  |Bangalore|Karnataka  |29 |Female|P102   |Active  |Complete

In [117]:
# 26
cust_payments = customers_clean.join(
    payments_clean, on='customer_id', how='left'
)
cust_payments.show(truncate=False)

+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+
|customer_id|customer_name|city     |state      |age|gender|plan_id|status  |data_quality_status|payment_id|bill_month         |amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+
|101        |Rahul Sharma |Hyderabad|Telangana  |35 |Male  |P101   |Active  |Complete           |5012      |2026-02-01 00:00:00|499        |Card        |Success       |Complete           |
|101        |Rahul Sharma |Hyderabad|Telangana  |35 |Male  |P101   |Active  |Complete           |5001      |2026-01-01 00:00:00|499        |UPI         |Success       |Complete           |
|102        |Priya Reddy  |Bangalore|Karnataka  |29 |Fe

In [118]:
# 27
complete_df = (
    customers_clean
    .join(usage_clean, on='customer_id', how='left')
    .join(payments_clean, on='customer_id', how='left')
    .join(plans_flat, on='plan_id', how='left')
)
complete_df.show(truncate=False)

+-------+-----------+-------------+---------+-----------+---+------+--------+-------------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|customer_id|customer_name|city     |state      |age|gender|status  |data_quality_status|usage_id|usage_month        |data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|bill_month         |amount_paid|payment_mode|payment_status|data_quality_status|plan_name   |monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming      |
+-------+-----------+-------------+---------+-----------+---+------+--------+-------------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+------------+--

In [119]:
# 28
invalid_plans = customers_clean.join(
    plans_flat, on='plan_id', how='left'
).filter(col('plan_name').isNull())
invalid_plans.select('customer_id', 'customer_name', 'plan_id').show()

+-----------+-------------+-------+
|customer_id|customer_name|plan_id|
+-----------+-------------+-------+
|        111|   Ravi Kumar|   P105|
|        112|  Ayesha Khan|UNKNOWN|
+-----------+-------------+-------+



In [120]:
# 29
usage_no_customer = usage_clean.join(
    customers_clean, on='customer_id', how='left'
).filter(col('customer_name').isNull())
usage_no_customer.show(truncate=False)

+-----------+--------+-------------------+------------+------------+---------+-------------------+-------------+----+-----+----+------+-------+------+-------------------+
|customer_id|usage_id|usage_month        |data_used_gb|call_minutes|sms_count|data_quality_status|customer_name|city|state|age |gender|plan_id|status|data_quality_status|
+-----------+--------+-------------------+------------+------------+---------+-------------------+-------------+----+-----+----+------+-------+------+-------------------+
|120        |1011    |2026-01-01 00:00:00|60          |1300        |140      |Complete           |NULL         |NULL|NULL |NULL|NULL  |NULL   |NULL  |NULL               |
+-----------+--------+-------------------+------------+------------+---------+-------------------+-------------+----+-----+----+------+-------+------+-------------------+



In [121]:
# 30
payments_no_customer = payments_clean.join(
    customers_clean, on='customer_id', how='left'
).filter(col('customer_name').isNull())
payments_no_customer.show(truncate=False)

+-----------+----------+----------+-----------+------------+--------------+-------------------+-------------+----+-----+---+------+-------+------+-------------------+
|customer_id|payment_id|bill_month|amount_paid|payment_mode|payment_status|data_quality_status|customer_name|city|state|age|gender|plan_id|status|data_quality_status|
+-----------+----------+----------+-----------+------------+--------------+-------------------+-------------+----+-----+---+------+-------+------+-------------------+
+-----------+----------+----------+-----------+------------+--------------+-------------------+-------------+----+-----+---+------+-------+------+-------------------+



In [122]:
# 31
usage_clean = usage_clean.withColumn(
    'usage_category',
    when(col('data_used_gb') >= 70, 'Heavy User')
    .when(col('data_used_gb') >= 30, 'Medium User')
    .otherwise('Low User')
)
usage_clean.select('customer_id', 'data_used_gb', 'usage_category').show()

+-----------+------------+--------------+
|customer_id|data_used_gb|usage_category|
+-----------+------------+--------------+
|        101|          45|   Medium User|
|        102|          30|   Medium User|
|        103|          12|      Low User|
|        104|          55|   Medium User|
|        105|          75|    Heavy User|
|        106|          28|      Low User|
|        107|          10|      Low User|
|        108|          80|    Heavy User|
|        109|          48|   Medium User|
|        110|          32|   Medium User|
|        120|          60|   Medium User|
|        101|          50|   Medium User|
|        102|          34|   Medium User|
|        104|          58|   Medium User|
|        105|           0|      Low User|
+-----------+------------+--------------+



In [123]:
# 32
payments_clean = payments_clean.withColumn(
    'payment_category',
    when(col('amount_paid') >= 1000, 'High Payment')
    .when(col('amount_paid') >= 500, 'Medium Payment')
    .otherwise('Low Payment')
)
payments_clean.select('customer_id', 'amount_paid', 'payment_category').show()

+-----------+-----------+----------------+
|customer_id|amount_paid|payment_category|
+-----------+-----------+----------------+
|        101|        499|     Low Payment|
|        102|        799|  Medium Payment|
|        103|        299|     Low Payment|
|        104|        499|     Low Payment|
|        105|       1199|    High Payment|
|        106|        799|  Medium Payment|
|        107|        299|     Low Payment|
|        108|       1199|    High Payment|
|        109|        499|     Low Payment|
|        110|        799|  Medium Payment|
|        112|          0|     Low Payment|
|        101|        499|     Low Payment|
|        102|        799|  Medium Payment|
|        104|        499|     Low Payment|
|        105|       1199|    High Payment|
+-----------+-----------+----------------+



In [124]:
# 33
complete_df = (
    customers_clean
    .join(usage_clean, on='customer_id', how='left')
    .join(payments_clean, on='customer_id', how='left')
    .join(plans_flat, on='plan_id', how='left')
).withColumn(
    'churn_risk',
    when(
        (col('status') == 'Inactive') | (col('payment_status') != 'Success'),
        'High Risk'
    )
    .when(col('data_used_gb') < 15, 'Medium Risk')
    .otherwise('Low Risk')
)
complete_df.select('customer_name', 'status', 'payment_status', 'data_used_gb', 'churn_risk').show()

+-------------+--------+--------------+------------+-----------+
|customer_name|  status|payment_status|data_used_gb| churn_risk|
+-------------+--------+--------------+------------+-----------+
| Rahul Sharma|  Active|       Success|          50|   Low Risk|
| Rahul Sharma|  Active|       Success|          50|   Low Risk|
| Rahul Sharma|  Active|       Success|          45|   Low Risk|
| Rahul Sharma|  Active|       Success|          45|   Low Risk|
|  Priya Reddy|  Active|       Success|          34|   Low Risk|
|  Priya Reddy|  Active|       Success|          34|   Low Risk|
|  Priya Reddy|  Active|       Success|          30|   Low Risk|
|  Priya Reddy|  Active|       Success|          30|   Low Risk|
|   Amit Kumar|Inactive|        Failed|          12|  High Risk|
|  Sneha Patel|  Active|       Success|          58|   Low Risk|
|  Sneha Patel|  Active|       Success|          58|   Low Risk|
|  Sneha Patel|  Active|       Success|          55|   Low Risk|
|  Sneha Patel|  Active| 

In [125]:
# 34
complete_df = complete_df.withColumn(
    'over_usage_gb',
    col('data_used_gb') - col('data_limit_gb')
)
complete_df.select('customer_name', 'data_used_gb', 'data_limit_gb', 'over_usage_gb').show()

+-------------+------------+-------------+-------------+
|customer_name|data_used_gb|data_limit_gb|over_usage_gb|
+-------------+------------+-------------+-------------+
| Rahul Sharma|          50|           50|            0|
| Rahul Sharma|          50|           50|            0|
| Rahul Sharma|          45|           50|           -5|
| Rahul Sharma|          45|           50|           -5|
|  Priya Reddy|          34|           75|          -41|
|  Priya Reddy|          34|           75|          -41|
|  Priya Reddy|          30|           75|          -45|
|  Priya Reddy|          30|           75|          -45|
|   Amit Kumar|          12|           25|          -13|
|  Sneha Patel|          58|           50|            8|
|  Sneha Patel|          58|           50|            8|
|  Sneha Patel|          55|           50|            5|
|  Sneha Patel|          55|           50|            5|
|   Farhan Ali|           0|          100|         -100|
|   Farhan Ali|           0|   

In [126]:
# 35
complete_df = complete_df.withColumn(
    'over_usage_flag',
    when(col('over_usage_gb') > 0, 'Yes').otherwise('No')
)
complete_df.select('customer_name', 'over_usage_gb', 'over_usage_flag').show()

+-------------+-------------+---------------+
|customer_name|over_usage_gb|over_usage_flag|
+-------------+-------------+---------------+
| Rahul Sharma|            0|             No|
| Rahul Sharma|            0|             No|
| Rahul Sharma|           -5|             No|
| Rahul Sharma|           -5|             No|
|  Priya Reddy|          -41|             No|
|  Priya Reddy|          -41|             No|
|  Priya Reddy|          -45|             No|
|  Priya Reddy|          -45|             No|
|   Amit Kumar|          -13|             No|
|  Sneha Patel|            8|            Yes|
|  Sneha Patel|            8|            Yes|
|  Sneha Patel|            5|            Yes|
|  Sneha Patel|            5|            Yes|
|   Farhan Ali|         -100|             No|
|   Farhan Ali|         -100|             No|
|   Farhan Ali|          -25|             No|
|   Farhan Ali|          -25|             No|
|   Neha Singh|          -47|             No|
|  Arjun Verma|          -15|     

In [127]:
# 36
customers_clean.groupBy('city').count().orderBy(col('count').desc()).show()

+---------+-----+
|     city|count|
+---------+-----+
|Hyderabad|    3|
|Bangalore|    2|
|   Mumbai|    2|
|    Delhi|    2|
|     Pune|    1|
|  Chennai|    1|
|    Kochi|    1|
+---------+-----+



In [128]:
# 37
customers_clean.groupBy('state').count().orderBy(col('count').desc()).show()

+-----------+-----+
|      state|count|
+-----------+-----+
|  Telangana|    3|
|Maharashtra|    3|
|  Karnataka|    2|
|      Delhi|    2|
| Tamil Nadu|    1|
|     Kerala|    1|
+-----------+-----+



In [129]:
# 38
customers_clean.groupBy('plan_id').count().show()

+-------+-----+
|plan_id|count|
+-------+-----+
|   P102|    3|
|   P103|    2|
|   P105|    1|
|UNKNOWN|    1|
|   P101|    3|
|   P104|    2|
+-------+-----+



In [130]:
# 39
usage_clean.groupBy('usage_category').count().show()

+--------------+-----+
|usage_category|count|
+--------------+-----+
|   Medium User|    9|
|      Low User|    4|
|    Heavy User|    2|
+--------------+-----+



In [131]:
# 40
complete_df.groupBy('churn_risk').count().show()

+-----------+-----+
| churn_risk|count|
+-----------+-----+
|  High Risk|    4|
|Medium Risk|    1|
|   Low Risk|   19|
+-----------+-----+



In [132]:
# 41
usage_clean.join(
    customers_clean.select('customer_id', 'plan_id'), on='customer_id'
).join(
    plans_flat.select('plan_id', 'plan_name'), on='plan_id'
).groupBy('plan_name').sum('data_used_gb').show()

+------------+-----------------+
|   plan_name|sum(data_used_gb)|
+------------+-----------------+
| Smart Basic|              256|
| Premium Max|              155|
|  Smart Plus|              124|
|Budget Saver|               22|
+------------+-----------------+



In [133]:
# 42
usage_clean.join(
    customers_clean.select('customer_id', 'plan_id'), on='customer_id'
).join(
    plans_flat.select('plan_id', 'plan_name'), on='plan_id'
).groupBy('plan_name').avg('data_used_gb').show()

+------------+------------------+
|   plan_name| avg(data_used_gb)|
+------------+------------------+
| Smart Basic|              51.2|
| Premium Max|51.666666666666664|
|  Smart Plus|              31.0|
|Budget Saver|              11.0|
+------------+------------------+



In [134]:
# 43
usage_clean.join(
    customers_clean.select('customer_id', 'city'), on='customer_id'
).groupBy('city').sum('call_minutes').show()

+---------+-----------------+
|     city|sum(call_minutes)|
+---------+-----------------+
|Hyderabad|             2100|
|Bangalore|             2200|
|     Pune|              500|
|   Mumbai|              250|
|  Chennai|             2300|
|    Delhi|             3650|
|    Kochi|             1600|
+---------+-----------------+



In [135]:
# 44
usage_clean.join(
    customers_clean.select('customer_id', 'state'), on='customer_id'
).groupBy('state').sum('sms_count').show()

+-----------+--------------+
|      state|sum(sms_count)|
+-----------+--------------+
|  Telangana|           270|
|Maharashtra|           100|
| Tamil Nadu|           310|
|  Karnataka|           265|
|      Delhi|           500|
|     Kerala|           250|
+-----------+--------------+



In [136]:
# 45
payments_clean.filter(
    col('payment_status') == 'Success'
).agg(sum('amount_paid').alias('total_revenue')).show()

+-------------+
|total_revenue|
+-------------+
|         8089|
+-------------+



In [137]:
# 46
payments_clean.join(
    customers_clean.select('customer_id', 'city'), on='customer_id'
).groupBy('city').sum('amount_paid').show()

+---------+----------------+
|     city|sum(amount_paid)|
+---------+----------------+
|Hyderabad|            1297|
|Bangalore|            2097|
|     Pune|             799|
|   Mumbai|             299|
|  Chennai|             998|
|    Delhi|            3197|
|    Kochi|            1199|
+---------+----------------+



In [138]:
# 47
payments_clean.join(
    customers_clean.select('customer_id', 'plan_id'), on='customer_id'
).join(
    plans_flat.select('plan_id', 'plan_name'), on='plan_id'
).groupBy('plan_name').sum('amount_paid').show()

+------------+----------------+
|   plan_name|sum(amount_paid)|
+------------+----------------+
| Smart Basic|            2495|
| Premium Max|            3597|
|  Smart Plus|            3196|
|Budget Saver|             598|
+------------+----------------+



In [139]:
# 48
payments_clean.groupBy('payment_mode').sum('amount_paid').show()

+------------+----------------+
|payment_mode|sum(amount_paid)|
+------------+----------------+
|        Card|            3696|
|         UPI|            4393|
|        Cash|             598|
|Not Provided|            1199|
+------------+----------------+



In [140]:
# 49
payments_clean.join(
    customers_clean.select('customer_id', 'plan_id'), on='customer_id'
).join(
    plans_flat.select('plan_id', 'plan_name'), on='plan_id'
).groupBy('plan_name').sum('amount_paid').orderBy(col('sum(amount_paid)').desc()).show(1)

+-----------+----------------+
|  plan_name|sum(amount_paid)|
+-----------+----------------+
|Premium Max|            3597|
+-----------+----------------+
only showing top 1 row


In [141]:
# 50
payments_clean.join(
    customers_clean.select('customer_id', 'city'), on='customer_id'
).groupBy('city').sum('amount_paid').orderBy(col('sum(amount_paid)').desc()).show(1)

+-----+----------------+
| city|sum(amount_paid)|
+-----+----------------+
|Delhi|            3197|
+-----+----------------+
only showing top 1 row


In [142]:
# 51
w = Window.orderBy(col('data_used_gb').desc())
(usage_clean
 .withColumn('usage_rank', rank().over(w))
 .select('customer_id', 'data_used_gb', 'usage_rank')
 .show())

+-----------+------------+----------+
|customer_id|data_used_gb|usage_rank|
+-----------+------------+----------+
|        108|          80|         1|
|        105|          75|         2|
|        120|          60|         3|
|        104|          58|         4|
|        104|          55|         5|
|        101|          50|         6|
|        109|          48|         7|
|        101|          45|         8|
|        102|          34|         9|
|        110|          32|        10|
|        102|          30|        11|
|        106|          28|        12|
|        103|          12|        13|
|        107|          10|        14|
|        105|           0|        15|
+-----------+------------+----------+



In [143]:
# 52
w = Window.orderBy(col('amount_paid').desc())
(payments_clean
 .withColumn('payment_rank', rank().over(w))
 .select('customer_id', 'amount_paid', 'payment_rank')
 .show())

+-----------+-----------+------------+
|customer_id|amount_paid|payment_rank|
+-----------+-----------+------------+
|        105|       1199|           1|
|        108|       1199|           1|
|        105|       1199|           1|
|        102|        799|           4|
|        106|        799|           4|
|        110|        799|           4|
|        102|        799|           4|
|        101|        499|           8|
|        104|        499|           8|
|        109|        499|           8|
|        101|        499|           8|
|        104|        499|           8|
|        103|        299|          13|
|        107|        299|          13|
|        112|          0|          15|
+-----------+-----------+------------+



In [144]:
# 53
w = Window.orderBy(col('data_used_gb').desc())
(usage_clean
 .withColumn('usage_rank', rank().over(w))
 .filter(col('usage_rank') <= 3)
 .select('customer_id', 'data_used_gb', 'usage_rank')
 .show())

+-----------+------------+----------+
|customer_id|data_used_gb|usage_rank|
+-----------+------------+----------+
|        108|          80|         1|
|        105|          75|         2|
|        120|          60|         3|
+-----------+------------+----------+



In [145]:
# 54
w = Window.orderBy(col('amount_paid').desc())
(payments_clean
 .withColumn('payment_rank', rank().over(w))
 .filter(col('payment_rank') <= 3)
 .select('customer_id', 'amount_paid', 'payment_rank')
 .show())

+-----------+-----------+------------+
|customer_id|amount_paid|payment_rank|
+-----------+-----------+------------+
|        105|       1199|           1|
|        108|       1199|           1|
|        105|       1199|           1|
+-----------+-----------+------------+



In [146]:
# 55
city_usage = usage_clean.join(
    customers_clean.select('customer_id', 'customer_name', 'city'), on='customer_id'
)
w = Window.partitionBy('city').orderBy(col('data_used_gb').desc())
(city_usage
 .withColumn('city_rank', rank().over(w))
 .filter(col('city_rank') == 1)
 .select('city', 'customer_name', 'data_used_gb')
 .show())

+---------+-------------+------------+
|     city|customer_name|data_used_gb|
+---------+-------------+------------+
|Bangalore|    Kiran Rao|          48|
|  Chennai|  Sneha Patel|          58|
|    Delhi|   Farhan Ali|          75|
|Hyderabad| Rahul Sharma|          50|
|    Kochi|   Meera Nair|          80|
|   Mumbai|   Amit Kumar|          12|
|     Pune|   Neha Singh|          28|
+---------+-------------+------------+



In [147]:
# 56
plan_usage = (
    usage_clean
    .join(customers_clean.select('customer_id', 'customer_name', 'plan_id'), on='customer_id')
    .join(plans_flat.select('plan_id', 'plan_name'), on='plan_id')
)
w = Window.partitionBy('plan_name').orderBy(col('data_used_gb').desc())
(plan_usage
 .withColumn('plan_rank', rank().over(w))
 .filter(col('plan_rank') == 1)
 .select('plan_name', 'customer_name', 'data_used_gb')
 .show())

+------------+-------------+------------+
|   plan_name|customer_name|data_used_gb|
+------------+-------------+------------+
|Budget Saver|   Amit Kumar|          12|
| Premium Max|   Meera Nair|          80|
| Smart Basic|  Sneha Patel|          58|
|  Smart Plus|  Priya Reddy|          34|
+------------+-------------+------------+



In [148]:
# 57
w = (
    Window
    .orderBy('bill_month')
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
(payments_clean
 .withColumn('running_revenue', sum('amount_paid').over(w))
 .select('payment_id', 'bill_month', 'amount_paid', 'running_revenue')
 .show())

+----------+-------------------+-----------+---------------+
|payment_id|         bill_month|amount_paid|running_revenue|
+----------+-------------------+-----------+---------------+
|      5001|2026-01-01 00:00:00|        499|            499|
|      5002|2026-01-01 00:00:00|        799|           1298|
|      5003|2026-01-01 00:00:00|        299|           1597|
|      5004|2026-01-01 00:00:00|        499|           2096|
|      5005|2026-01-01 00:00:00|       1199|           3295|
|      5006|2026-01-01 00:00:00|        799|           4094|
|      5007|2026-01-01 00:00:00|        299|           4393|
|      5008|2026-01-01 00:00:00|       1199|           5592|
|      5009|2026-01-01 00:00:00|        499|           6091|
|      5010|2026-01-01 00:00:00|        799|           6890|
|      5011|2026-01-01 00:00:00|          0|           6890|
|      5012|2026-02-01 00:00:00|        499|           7389|
|      5013|2026-02-01 00:00:00|        799|           8188|
|      5014|2026-02-01 0

In [149]:
# 58
w = Window.partitionBy('customer_id').orderBy('usage_month')
(usage_clean
 .withColumn('prev_data', lag('data_used_gb', 1).over(w))
 .select('customer_id', 'usage_month', 'data_used_gb', 'prev_data')
 .show())

+-----------+-------------------+------------+---------+
|customer_id|        usage_month|data_used_gb|prev_data|
+-----------+-------------------+------------+---------+
|        101|2026-01-01 00:00:00|          45|     NULL|
|        101|2026-02-01 00:00:00|          50|       45|
|        102|2026-01-01 00:00:00|          30|     NULL|
|        102|2026-02-01 00:00:00|          34|       30|
|        103|2026-01-01 00:00:00|          12|     NULL|
|        104|2026-01-01 00:00:00|          55|     NULL|
|        104|2026-02-01 00:00:00|          58|       55|
|        105|2026-01-01 00:00:00|          75|     NULL|
|        105|2026-02-01 00:00:00|           0|       75|
|        106|2026-01-01 00:00:00|          28|     NULL|
|        107|2026-01-01 00:00:00|          10|     NULL|
|        108|2026-01-01 00:00:00|          80|     NULL|
|        109|2026-01-01 00:00:00|          48|     NULL|
|        110|2026-01-01 00:00:00|          32|     NULL|
|        120|2026-01-01 00:00:0

In [150]:
# 59
w = Window.partitionBy('customer_id').orderBy('usage_month')
(usage_clean
 .withColumn('next_data', lead('data_used_gb', 1).over(w))
 .select('customer_id', 'usage_month', 'data_used_gb', 'next_data')
 .show())

+-----------+-------------------+------------+---------+
|customer_id|        usage_month|data_used_gb|next_data|
+-----------+-------------------+------------+---------+
|        101|2026-01-01 00:00:00|          45|       50|
|        101|2026-02-01 00:00:00|          50|     NULL|
|        102|2026-01-01 00:00:00|          30|       34|
|        102|2026-02-01 00:00:00|          34|     NULL|
|        103|2026-01-01 00:00:00|          12|     NULL|
|        104|2026-01-01 00:00:00|          55|       58|
|        104|2026-02-01 00:00:00|          58|     NULL|
|        105|2026-01-01 00:00:00|          75|        0|
|        105|2026-02-01 00:00:00|           0|     NULL|
|        106|2026-01-01 00:00:00|          28|     NULL|
|        107|2026-01-01 00:00:00|          10|     NULL|
|        108|2026-01-01 00:00:00|          80|     NULL|
|        109|2026-01-01 00:00:00|          48|     NULL|
|        110|2026-01-01 00:00:00|          32|     NULL|
|        120|2026-01-01 00:00:0

In [151]:
# 60
w = Window.partitionBy('customer_id').orderBy('usage_month')
(usage_clean
 .withColumn('prev_data', lag('data_used_gb', 1).over(w))
 .filter(col('data_used_gb') > col('prev_data'))
 .select('customer_id', 'usage_month', 'prev_data', 'data_used_gb')
 .show())

+-----------+-------------------+---------+------------+
|customer_id|        usage_month|prev_data|data_used_gb|
+-----------+-------------------+---------+------------+
|        101|2026-02-01 00:00:00|       45|          50|
|        102|2026-02-01 00:00:00|       30|          34|
|        104|2026-02-01 00:00:00|       55|          58|
+-----------+-------------------+---------+------------+



In [152]:
# 61
customers_clean.createOrReplaceTempView("customers")
usage_clean.createOrReplaceTempView("usage")
payments_clean.createOrReplaceTempView("payments")
plans_flat.createOrReplaceTempView("plans")
print("Temp views created")

Temp views created


In [153]:
# 62
spark.sql("SELECT * FROM customers WHERE status = 'Active'").show(truncate=False)

+-----------+-------------+---------+-----------+---+------+-------+------+-------------------+
|customer_id|customer_name|city     |state      |age|gender|plan_id|status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+------+-------------------+
|101        |Rahul Sharma |Hyderabad|Telangana  |35 |Male  |P101   |Active|Complete           |
|102        |Priya Reddy  |Bangalore|Karnataka  |29 |Female|P102   |Active|Complete           |
|104        |Sneha Patel  |Chennai  |Tamil Nadu |31 |Female|P101   |Active|Complete           |
|105        |Farhan Ali   |Delhi    |Delhi      |55 |Male  |P104   |Active|Complete           |
|106        |Neha Singh   |Pune     |Maharashtra|38 |Female|P102   |Active|Complete           |
|108        |Meera Nair   |Kochi    |Kerala     |48 |Female|P104   |Active|Complete           |
|109        |Kiran Rao    |Bangalore|Karnataka  |33 |Male  |P101   |Active|Complete           |
|110        |Nisha Reddy  |Delhi    |Del

In [154]:
# 63
spark.sql('SELECT city, COUNT(*) as total FROM customers GROUP BY city ORDER BY total DESC').show()

+---------+-----+
|     city|total|
+---------+-----+
|Hyderabad|    3|
|Bangalore|    2|
|   Mumbai|    2|
|    Delhi|    2|
|     Pune|    1|
|  Chennai|    1|
|    Kochi|    1|
+---------+-----+



In [155]:
# 64
spark.sql("""
    SELECT p.plan_name, SUM(pay.amount_paid) as revenue
    FROM payments pay
    JOIN customers c ON pay.customer_id = c.customer_id
    JOIN plans p ON c.plan_id = p.plan_id
    GROUP BY p.plan_name
    ORDER BY revenue DESC
""").show()

+------------+-------+
|   plan_name|revenue|
+------------+-------+
| Premium Max|   3597|
|  Smart Plus|   3196|
| Smart Basic|   2495|
|Budget Saver|    598|
+------------+-------+



In [156]:
# 65
spark.sql("SELECT * FROM usage WHERE data_used_gb >= 70").show()

+--------+-----------+-------------------+------------+------------+---------+-------------------+--------------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|usage_category|
+--------+-----------+-------------------+------------+------------+---------+-------------------+--------------+
|    1005|        105|2026-01-01 00:00:00|          75|        1500|      200|           Complete|    Heavy User|
|    1008|        108|2026-01-01 00:00:00|          80|        1600|      250|           Complete|    Heavy User|
+--------+-----------+-------------------+------------+------------+---------+-------------------+--------------+



In [157]:
# 66
spark.sql("""
    SELECT c.customer_id, c.customer_name, c.status, pay.payment_status
    FROM customers c
    JOIN payments pay ON c.customer_id = pay.customer_id
    WHERE c.status = 'Inactive' OR pay.payment_status != 'Success'
""").show(truncate=False)

+-----------+-------------+--------+--------------+
|customer_id|customer_name|status  |payment_status|
+-----------+-------------+--------+--------------+
|103        |Amit Kumar   |Inactive|Failed        |
|105        |Farhan Ali   |Active  |Pending       |
|107        |Arjun Verma  |Inactive|Pending       |
+-----------+-------------+--------+--------------+



In [158]:
# 67
spark.sql("""
    SELECT c.customer_id, c.customer_name, c.plan_id
    FROM customers c
    LEFT JOIN plans p ON c.plan_id = p.plan_id
    WHERE p.plan_name IS NULL
""").show()

+-----------+-------------+-------+
|customer_id|customer_name|plan_id|
+-----------+-------------+-------+
|        111|   Ravi Kumar|   P105|
|        112|  Ayesha Khan|UNKNOWN|
+-----------+-------------+-------+



In [159]:
# 68
spark.sql("SELECT * FROM payments WHERE payment_status IN ('Failed','Pending')").show()

+----------+-----------+-------------------+-----------+------------+--------------+-------------------+----------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|payment_category|
+----------+-----------+-------------------+-----------+------------+--------------+-------------------+----------------+
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|           Complete|     Low Payment|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|           Complete|     Low Payment|
|      5015|        105|2026-02-01 00:00:00|       1199|Not Provided|       Pending|           Complete|    High Payment|
+----------+-----------+-------------------+-----------+------------+--------------+-------------------+----------------+



In [160]:
# 69
spark.sql('SELECT customer_id, data_used_gb FROM usage ORDER BY data_used_gb DESC LIMIT 5').show()

+-----------+------------+
|customer_id|data_used_gb|
+-----------+------------+
|        108|          80|
|        105|          75|
|        120|          60|
|        104|          58|
|        104|          55|
+-----------+------------+



In [161]:
# 70
spark.sql('SELECT payment_mode, SUM(amount_paid) as revenue FROM payments GROUP BY payment_mode').show()

+------------+-------+
|payment_mode|revenue|
+------------+-------+
|        Card|   3696|
|         UPI|   4393|
|        Cash|    598|
|Not Provided|   1199|
+------------+-------+



In [162]:
# 71
complete_df_clean = (
    customers_clean.drop('data_quality_status')
    .join(usage_clean.drop('data_quality_status'), on='customer_id', how='left')
    .join(payments_clean.drop('data_quality_status'), on='customer_id', how='left')
    .join(plans_flat, on='plan_id', how='left')
)
complete_df_clean.write.mode("overwrite").parquet(GOLD + r"\complete")
print("Gold output saved")

Gold output saved


In [163]:
# 72
complete_df_clean = (
    customers_clean.drop('data_quality_status')
    .join(usage_clean.drop('data_quality_status'), on='customer_id', how='left')
    .join(payments_clean.drop('data_quality_status'), on='customer_id', how='left')
    .join(plans_flat, on='plan_id', how='left')
)
complete_df_clean.write.mode("overwrite").partitionBy("usage_month").parquet(GOLD + r"\partitioned")
print("Gold partitioned output saved")

Gold partitioned output saved


In [164]:
# 73
incremental_data = """usage_id,customer_id,usage_month,data_used_gb,call_minutes,sms_count
1016,101,2026-03,60,1100,140
1017,102,2026-03,40,700,90
1018,104,2026-03,70,1300,180"""

with open(r"C:\new_env\hexaware-sql\june_13\capstone_1\usage_incremental.csv", "w") as f:
    f.write(incremental_data)
print("Incremental file created")

Incremental file created


In [165]:
# 74
usage_inc_df = spark.read.csv(
    r"C:\new_env\hexaware-sql\june_13\capstone_1\usage_incremental.csv",
    header=True, inferSchema=True
)
usage_inc_df.show()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1016|        101|2026-03-01 00:00:00|          60|        1100|      140|
|    1017|        102|2026-03-01 00:00:00|          40|         700|       90|
|    1018|        104|2026-03-01 00:00:00|          70|        1300|      180|
+--------+-----------+-------------------+------------+------------+---------+



In [166]:
# 75
usage_inc_clean = usage_inc_df.fillna({'data_used_gb': 0})
usage_inc_clean.write.mode("append").parquet(SILVER + r"\usage")
print("Incremental data appended")

Incremental data appended


In [167]:
# 76
usage_full = spark.read.parquet(SILVER + r"\usage")
usage_summary = (
    usage_full
    .join(customers_clean.select('customer_id', 'customer_name'), on='customer_id')
    .groupBy('customer_id', 'customer_name')
    .agg(
        sum('data_used_gb').alias('total_data'),
        sum('call_minutes').alias('total_calls'),
        sum('sms_count').alias('total_sms')
    )
)
usage_summary.show()

+-----------+-------------+----------+-----------+---------+
|customer_id|customer_name|total_data|total_calls|total_sms|
+-----------+-------------+----------+-----------+---------+
|        105|   Farhan Ali|        75|       2950|      410|
|        107|  Arjun Verma|        10|        200|       20|
|        108|   Meera Nair|        80|       1600|      250|
|        109|    Kiran Rao|        48|        950|      100|
|        110|  Nisha Reddy|        32|        700|       90|
|        101| Rahul Sharma|       155|       3000|      390|
|        102|  Priya Reddy|       104|       1950|      255|
|        103|   Amit Kumar|        12|        250|       40|
|        104|  Sneha Patel|       183|       3600|      490|
|        106|   Neha Singh|        28|        500|       60|
+-----------+-------------+----------+-----------+---------+



In [168]:
# 77
usage_full = spark.read.parquet(SILVER + r"\usage").drop('data_quality_status')

complete_new = (
    customers_clean.drop('data_quality_status')
    .join(usage_full, on='customer_id', how='left')
    .join(payments_clean.drop('data_quality_status'), on='customer_id', how='left')
    .join(plans_flat, on='plan_id', how='left')
)
complete_new.write.mode("overwrite").partitionBy("usage_month").parquet(GOLD + r"\partitioned")
print("Gold re-partitioned")

Gold re-partitioned


In [169]:
# 78
before = usage_df.count()
after = spark.read.parquet(SILVER + r"\usage").count()
print(f"Before incremental load : {before}")
print(f"After  incremental load : {after}")
print(f"New records added       : {after - before}")

Before incremental load : 15
After  incremental load : 18
New records added       : 3


In [170]:
# 79
gold_79 = (
    customers_clean
    .join(usage_clean, on='customer_id', how='left')
    .join(payments_clean, on='customer_id', how='left')
    .join(plans_flat, on='plan_id', how='left')
).withColumn(
    'churn_risk',
    when(
        (col('status') == 'Inactive') | (col('payment_status') != 'Success'),
        'High Risk'
    )
    .when(col('data_used_gb') < 15, 'Medium Risk')
    .otherwise('Low Risk')
).withColumn(
    'over_usage_flag',
    when((col('data_used_gb') - col('data_limit_gb')) > 0, 'Yes').otherwise('No')
).select(
    'customer_id', 'customer_name', 'city', 'plan_name',
    'usage_month', 'data_used_gb', 'data_limit_gb',
    'over_usage_flag', 'amount_paid', 'payment_status', 'churn_risk'
)
gold_79.show(truncate=False)
gold_79.write.mode("overwrite").parquet(GOLD + r"\customer_usage_summary")

+-----------+-------------+---------+------------+-------------------+------------+-------------+---------------+-----------+--------------+-----------+
|customer_id|customer_name|city     |plan_name   |usage_month        |data_used_gb|data_limit_gb|over_usage_flag|amount_paid|payment_status|churn_risk |
+-----------+-------------+---------+------------+-------------------+------------+-------------+---------------+-----------+--------------+-----------+
|101        |Rahul Sharma |Hyderabad|Smart Basic |2026-02-01 00:00:00|50          |50           |No             |499        |Success       |Low Risk   |
|101        |Rahul Sharma |Hyderabad|Smart Basic |2026-02-01 00:00:00|50          |50           |No             |499        |Success       |Low Risk   |
|101        |Rahul Sharma |Hyderabad|Smart Basic |2026-01-01 00:00:00|45          |50           |No             |499        |Success       |Low Risk   |
|101        |Rahul Sharma |Hyderabad|Smart Basic |2026-01-01 00:00:00|45          

In [171]:
# 80
gold_80 = (
    usage_clean
    .join(customers_clean.select('customer_id', 'plan_id'), on='customer_id')
    .join(plans_flat, on='plan_id')
    .join(payments_clean.select('customer_id', 'amount_paid'), on='customer_id')
    .groupBy('plan_name')
    .agg(
        count('customer_id').alias('total_customers'),
        sum('data_used_gb').alias('total_data_usage'),
        avg('data_used_gb').alias('average_data_usage'),
        sum('amount_paid').alias('total_revenue')
    )
)
gold_80.show(truncate=False)
gold_80.write.mode("overwrite").parquet(GOLD + r"\plan_performance")

+------------+---------------+----------------+------------------+-------------+
|plan_name   |total_customers|total_data_usage|average_data_usage|total_revenue|
+------------+---------------+----------------+------------------+-------------+
|Smart Basic |9              |464             |51.55555555555556 |4491         |
|Premium Max |5              |230             |46.0              |5995         |
|Smart Plus  |6              |188             |31.333333333333332|4794         |
|Budget Saver|2              |22              |11.0              |598          |
+------------+---------------+----------------+------------------+-------------+



In [172]:
# 81
gold_81 = (
    payments_clean
    .join(customers_clean.select('customer_id', 'city'), on='customer_id')
    .groupBy('city')
    .agg(
        count('customer_id').alias('total_customers'),
        sum('amount_paid').alias('total_revenue'),
        avg('amount_paid').alias('average_payment')
    )
)
gold_81.show(truncate=False)
gold_81.write.mode("overwrite").parquet(GOLD + r"\city_revenue")

+---------+---------------+-------------+------------------+
|city     |total_customers|total_revenue|average_payment   |
+---------+---------------+-------------+------------------+
|Hyderabad|4              |1297         |324.25            |
|Bangalore|3              |2097         |699.0             |
|Pune     |1              |799          |799.0             |
|Mumbai   |1              |299          |299.0             |
|Chennai  |2              |998          |499.0             |
|Delhi    |3              |3197         |1065.6666666666667|
|Kochi    |1              |1199         |1199.0            |
+---------+---------------+-------------+------------------+



In [173]:
# 82
gold_82 = (
    customers_clean
    .join(payments_clean.select('customer_id', 'payment_status'), on='customer_id')
    .join(usage_clean.select('customer_id', 'data_used_gb'), on='customer_id')
    .join(plans_flat.select('plan_id', 'plan_name'), on='plan_id')
).withColumn(
    'churn_risk',
    when(
        (col('status') == 'Inactive') | (col('payment_status') != 'Success'),
        'High Risk'
    )
    .when(col('data_used_gb') < 15, 'Medium Risk')
    .otherwise('Low Risk')
).select(
    'customer_id', 'customer_name', 'city',
    'plan_name', 'payment_status', 'status', 'churn_risk'
)
gold_82.show(truncate=False)
gold_82.write.mode("overwrite").parquet(GOLD + r"\churn_risk")

+-----------+-------------+---------+------------+--------------+--------+-----------+
|customer_id|customer_name|city     |plan_name   |payment_status|status  |churn_risk |
+-----------+-------------+---------+------------+--------------+--------+-----------+
|101        |Rahul Sharma |Hyderabad|Smart Basic |Success       |Active  |Low Risk   |
|101        |Rahul Sharma |Hyderabad|Smart Basic |Success       |Active  |Low Risk   |
|101        |Rahul Sharma |Hyderabad|Smart Basic |Success       |Active  |Low Risk   |
|101        |Rahul Sharma |Hyderabad|Smart Basic |Success       |Active  |Low Risk   |
|102        |Priya Reddy  |Bangalore|Smart Plus  |Success       |Active  |Low Risk   |
|102        |Priya Reddy  |Bangalore|Smart Plus  |Success       |Active  |Low Risk   |
|102        |Priya Reddy  |Bangalore|Smart Plus  |Success       |Active  |Low Risk   |
|102        |Priya Reddy  |Bangalore|Smart Plus  |Success       |Active  |Low Risk   |
|103        |Amit Kumar   |Mumbai   |Budget

In [174]:
# 83
gold_83 = (
    usage_clean
    .join(customers_clean.select('customer_id', 'customer_name', 'plan_id'), on='customer_id')
    .join(plans_flat.select('plan_id', 'plan_name', 'data_limit_gb'), on='plan_id')
).withColumn(
    'over_usage_gb',
    col('data_used_gb') - col('data_limit_gb')
).filter(
    col('over_usage_gb') > 0
).select(
    'customer_id', 'customer_name', 'plan_name',
    'data_used_gb', 'data_limit_gb', 'over_usage_gb'
)
gold_83.show(truncate=False)
gold_83.write.mode("overwrite").parquet(GOLD + r"\over_usage")
print("All Gold Reports saved!")

+-----------+-------------+-----------+------------+-------------+-------------+
|customer_id|customer_name|plan_name  |data_used_gb|data_limit_gb|over_usage_gb|
+-----------+-------------+-----------+------------+-------------+-------------+
|104        |Sneha Patel  |Smart Basic|58          |50           |8            |
|104        |Sneha Patel  |Smart Basic|55          |50           |5            |
+-----------+-------------+-----------+------------+-------------+-------------+

All Gold Reports saved!
